<a href="https://colab.research.google.com/github/uzair848/AI/blob/main/1BM24CS144_8_puzzle_using_A*(week3)_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import heapq

class Node:
    def __init__(self, state, parent=None, action=None, g=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g

    def __lt__(self, other):
        return self.g < other.g

class EightPuzzleProblem:
    def __init__(self, initial_state, goal_state):
        self.initialState = initial_state
        self.goal_state = goal_state

    def goalTest(self, state):
        return state == self.goal_state

    def actions(self, state):
        valid_actions = []
        r, c = next((r, c) for r in range(3) for c in range(3) if state[r][c] == 0)

        if r > 0: valid_actions.append('Up')
        if r < 2: valid_actions.append('Down')
        if c > 0: valid_actions.append('Left')
        if c < 2: valid_actions.append('Right')

        return valid_actions

    def transition(self, state, action):
        r, c = next((r, c) for r in range(3) for c in range(3) if state[r][c] == 0)
        new_board = [list(row) for row in state]

        if action == 'Up':
            new_board[r][c], new_board[r-1][c] = new_board[r-1][c], new_board[r][c]
        elif action == 'Down':
            new_board[r][c], new_board[r+1][c] = new_board[r+1][c], new_board[r][c]
        elif action == 'Left':
            new_board[r][c], new_board[r][c-1] = new_board[r][c-1], new_board[r][c]
        elif action == 'Right':
            new_board[r][c], new_board[r][c+1] = new_board[r][c+1], new_board[r][c]

        return tuple(tuple(row) for row in new_board)


def manhattan_heuristic(state, goal_state):
    distance = 0
    goal_positions = {}
    for r in range(3):
        for c in range(3):
            goal_positions[goal_state[r][c]] = (r, c)

    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:
                target_r, target_c = goal_positions[val]
                distance += abs(r - target_r) + abs(c - target_c)
    return distance


def get_solution_nodes(node):
    nodes_path = []
    current = node
    while current is not None:
        nodes_path.append(current)
        current = current.parent
    nodes_path.reverse()
    return nodes_path


def a_star_search(problem):
    root = Node(state=problem.initialState, parent=None, action=None, g=0)
    frontier = []
    h_initial = manhattan_heuristic(root.state, problem.goal_state)
    heapq.heappush(frontier, (root.g + h_initial, root))
    reached = {root.state: root.g}

    while True:
        if not frontier:
            return None

        f, n = heapq.heappop(frontier)

        if problem.goalTest(n.state):
            return get_solution_nodes(n)

        for action in problem.actions(n.state):
            child_state = problem.transition(n.state, action)
            child_g = n.g + 1

            if child_state not in reached or child_g < reached[child_state]:
                reached[child_state] = child_g
                child_node = Node(state=child_state, parent=n, action=action, g=child_g)
                f_child = child_g + manhattan_heuristic(child_state, problem.goal_state)
                heapq.heappush(frontier, (f_child, child_node))


def get_matrix_input(label):
    print(f"\n--- Enter the {label} Board Configuration ---")
    print("Provide 3 space-separated numbers per row (0 represents the blank tile):")
    board = []
    for i in range(3):
        while True:
            try:
                row_input = input(f"Row {i+1}: ").strip().split()
                row = [int(x) for x in row_input]
                if len(row) != 3:
                    raise ValueError("Each row must have exactly 3 numbers.")
                board.append(tuple(row))
                break
            except ValueError as e:
                print(f"Invalid input: {e}. Please re-enter the row.")
    return tuple(board)


def print_state(state):
    for row in state:
        print("  ".join(map(str, row)))
    print()


if __name__ == "__main__":
    initial_board = get_matrix_input("Initial")
    goal_board = get_matrix_input("Final (Goal)")

    puzzle_problem = EightPuzzleProblem(initial_board, goal_board)

    print("\nSearching for an optimal path using A* Search...")
    solution_nodes = a_star_search(puzzle_problem)

    if solution_nodes is not None:
        print(f"\n🎉 Success! Solution found in {len(solution_nodes) - 1} moves.\n")

        for index, node in enumerate(solution_nodes):
            if index == 0:
                print("Initial State:")
            else:
                print(f"Move {index}: Step Moved -> {node.action}")
            print_state(node.state)
    else:
        print("\n❌ Failure: This configuration translation is unsolvable.")



--- Enter the Initial Board Configuration ---
Provide 3 space-separated numbers per row (0 represents the blank tile):
Row 1: 2 8 3
Row 2: 1 6 4
Row 3: 7 0 5

--- Enter the Final (Goal) Board Configuration ---
Provide 3 space-separated numbers per row (0 represents the blank tile):
Row 1: 1 2 3
Row 2: 8 0 4
Row 3: 7 6 5

Searching for an optimal path using A* Search...

🎉 Success! Solution found in 5 moves.

Initial State:
2  8  3
1  6  4
7  0  5

Move 1: Step Moved -> Up
2  8  3
1  0  4
7  6  5

Move 2: Step Moved -> Up
2  0  3
1  8  4
7  6  5

Move 3: Step Moved -> Left
0  2  3
1  8  4
7  6  5

Move 4: Step Moved -> Down
1  2  3
0  8  4
7  6  5

Move 5: Step Moved -> Right
1  2  3
8  0  4
7  6  5

